In [62]:
from pathlib import Path
import pandas as pd

In [63]:
RAW = Path("..") / "data" / "raw"
for folder in sorted(RAW.iterdir()):
    print(folder.name, "→", len(list(folder.rglob("*"))), "files")

.DS_Store → 0 files
ddi → 657 files
fitzpatrick17k → 16580 files
isic → 25333 files


In [64]:
for csv in RAW.rglob("*.csv"):
    cols = pd.read_csv(csv, nrows=0).columns.tolist()
    print(csv.relative_to(RAW), "→", cols[:8])

isic/metadata.csv → ['isic_id', 'attribution', 'copyright_license', 'age_approx', 'anatom_site_1', 'anatom_site_2', 'anatom_site_3', 'anatom_site_4']
ddi/ddi_metadata.csv → ['Unnamed: 0', 'DDI_ID', 'DDI_file', 'skin_tone', 'malignant', 'disease']
fitzpatrick17k/fitzpatrick17ku.csv → ['orig_img_name', 'new_img_name', 'md5hash', 'fitzpatrick', 'label', 'nine_partition_label', 'three_partition_label', 'qc']
fitzpatrick17k/fitzpatrick17k.csv → ['md5hash', 'fitzpatrick_scale', 'fitzpatrick_centaur', 'label', 'nine_partition_label', 'three_partition_label', 'qc', 'url']


In [65]:
ddi = pd.read_csv(RAW / "ddi" / "ddi_metadata.csv", index_col = 0)
print(len(ddi))
print(ddi.columns.tolist())

656
['DDI_ID', 'DDI_file', 'skin_tone', 'malignant', 'disease']


In [66]:
print(ddi.shape)

(656, 5)


In [67]:
ddi.head()

,DDI_ID,DDI_file,skin_tone,malignant,disease
0,1,000001.png,56,True,melanoma-in-situ
1,2,000002.png,56,True,melanoma-in-situ
2,3,000003.png,56,True,mycosis-fungoides
3,4,000004.png,56,True,squamous-cell-carcinoma-in-situ
4,5,000005.png,12,True,basal-cell-carcinoma


In [68]:
ddi["skin_tone"].value_counts()

skin_tone
34    241
12    208
56    207
Name: count, dtype: int64

In [69]:
pd.crosstab(ddi["skin_tone"], ddi["malignant"])

malignant,False,True
skin_tone,,
12,159,49
34,167,74
56,159,48


In [70]:
on_disk = {p.name for p in (RAW / "ddi").rglob("*.png")}
ddi["DDI_file"].isin(on_disk).mean()

np.float64(1.0)

In [71]:
fitz_raw = pd.read_csv(RAW / "fitzpatrick17k" / "fitzpatrick17ku.csv") # never touched again
print(len(fitz_raw))
fitz = fitz_raw[fitz_raw["fitzpatrick"] != -1].copy()
print(len(fitz))
print(fitz.columns.tolist())

16577
16012
['orig_img_name', 'new_img_name', 'md5hash', 'fitzpatrick', 'label', 'nine_partition_label', 'three_partition_label', 'qc', 'url', 'url_alphanum']


In [72]:
print(fitz.shape)

(16012, 10)


In [73]:
fitz.head()

,orig_img_name,new_img_name,md5hash,fitzpatrick,label,nine_partition_label,three_partition_label,qc,url,url_alphanum
0,5e82a45bc5d78bd24ae9202d194423f8.jpg,dr-in-pi-ch_f3_0_5e82a45b.jpg,5e82a45bc5d78bd24ae9202d194423f8,3,drug induced pigmentary changes,inflammatory,non-neoplastic,NaN,https://www.dermaamin.com/site/images/clinical...,httpwwwdermaamincomsiteimagesclinicalpicmminoc...
1,fa2911a9b13b6f8af79cb700937cc14f.jpg,ph_f1_0_fa2911a9.jpg,fa2911a9b13b6f8af79cb700937cc14f,1,photodermatoses,inflammatory,non-neoplastic,NaN,https://www.dermaamin.com/site/images/clinical...,httpwwwdermaamincomsiteimagesclinicalpicpphoto...
2,d2bac3c9e4499032ca8e9b07c7d3bc40.jpg,defi_f2_0_d2bac3c9.jpg,d2bac3c9e4499032ca8e9b07c7d3bc40,2,dermatofibroma,benign dermal,benign,NaN,https://www.dermaamin.com/site/images/clinical...,httpwwwdermaamincomsiteimagesclinicalpicdderma...
3,0a94359e7eaacd7178e06b2823777789.jpg,ps_f1_0_0a94359e.jpg,0a94359e7eaacd7178e06b2823777789,1,psoriasis,inflammatory,non-neoplastic,NaN,https://www.dermaamin.com/site/images/clinical...,httpwwwdermaamincomsiteimagesclinicalpicppsori...
4,a39ec3b1f22c08a421fa20535e037bba.jpg,ps_f1_1_a39ec3b1.jpg,a39ec3b1f22c08a421fa20535e037bba,1,psoriasis,inflammatory,non-neoplastic,NaN,https://www.dermaamin.com/site/images/clinical...,httpwwwdermaamincomsiteimagesclinicalpicppsori...


In [74]:
fitz["fitzpatrick"].value_counts()

fitzpatrick
2    4808
3    3308
1    2947
4    2781
5    1533
6     635
Name: count, dtype: int64

In [75]:
pd.crosstab(fitz["fitzpatrick"], fitz["three_partition_label"])

three_partition_label,benign,malignant,non-neoplastic
fitzpatrick,,,
1,444,453,2050
2,671,742,3395
3,475,456,2377
4,367,301,2113
5,159,147,1227
6,44,61,530


In [76]:
on_disk = {p.name for p in (RAW / "fitzpatrick17k").rglob("*.jpg")}
fitz["orig_img_name"].isin(on_disk).mean()   # fraction of register rows whose image exists

np.float64(1.0)

In [77]:
on_disk = {p.name for p in (RAW / "fitzpatrick17k").rglob("*.jpg")}
(fitz["md5hash"] + ".jpg").isin(on_disk).mean()   # fraction of register rows whose image exists

np.float64(1.0)

In [78]:
(fitz["orig_img_name"] == fitz["md5hash"] + ".jpg").mean()

np.float64(1.0)

In [79]:
len(on_disk), len(fitz)

(16577, 16012)

In [80]:
print(list(on_disk)[:3])                # what do real filenames look like?
print(fitz["new_img_name"].head(3).tolist()) # what does the register say?
print(fitz["md5hash"].head(3).tolist())

['0564e32394d14c2c16b4b1e35def1711.jpg', '0f119ca91147595c6299c830803cfb47.jpg', '45321a9a4b3fa9f5f4a132f87d63e9df.jpg']
['dr-in-pi-ch_f3_0_5e82a45b.jpg', 'ph_f1_0_fa2911a9.jpg', 'defi_f2_0_d2bac3c9.jpg']
['5e82a45bc5d78bd24ae9202d194423f8', 'fa2911a9b13b6f8af79cb700937cc14f', 'd2bac3c9e4499032ca8e9b07c7d3bc40']


In [81]:
fitz["md5hash"].is_unique

True

In [82]:
# Keep only rows where three_partition_label is benign or malignant
fitz = fitz[fitz["three_partition_label"].isin(["benign", "malignant"])]
print(len(fitz))

4320


In [83]:
# Map fitzpatrick to light / medium / dark with your dictionary
skin_map = {1: "light", 2: "light", 3: "medium", 4: "medium", 5: "dark", 6: "dark"}
fitz["skin_group"] = fitz["fitzpatrick"].map(skin_map)
print(fitz["skin_group"].value_counts(dropna=False))
print(len(fitz))

skin_group
light     2310
medium    1599
dark       411
Name: count, dtype: int64
4320


In [84]:
# Cross-tabulate skin group against benign/malignant
pd.crosstab(fitz["skin_group"], fitz["three_partition_label"]).reindex(["light", "medium", "dark"])

three_partition_label,benign,malignant
skin_group,,
light,1115,1195
medium,842,757
dark,203,208


In [85]:
isic_raw = pd.read_csv(RAW / "isic" / "metadata.csv")
print(len(isic_raw))
print(isic_raw.columns.tolist())

25331
['isic_id', 'attribution', 'copyright_license', 'age_approx', 'anatom_site_1', 'anatom_site_2', 'anatom_site_3', 'anatom_site_4', 'anatom_site_5', 'anatom_site_special', 'clin_size_long_diam_mm', 'concomitant_biopsy', 'dermoscopic_type', 'diagnosis_1', 'diagnosis_2', 'diagnosis_3', 'diagnosis_4', 'diagnosis_5', 'diagnosis_confirm_type', 'family_hx_mm', 'image_type', 'lesion_id', 'melanocytic', 'patient_id', 'personal_hx_mm', 'sex']


In [86]:
print(isic_raw.shape)

(25331, 26)


In [87]:
isic_raw.head()

,isic_id,attribution,copyright_license,age_approx,anatom_site_1,anatom_site_2,anatom_site_3,anatom_site_4,anatom_site_5,anatom_site_special,...,diagnosis_4,diagnosis_5,diagnosis_confirm_type,family_hx_mm,image_type,lesion_id,melanocytic,patient_id,personal_hx_mm,sex
0,ISIC_0000000,Anonymous,CC-0,55.0,Trunk,Anterior trunk,Anterior abdomen,NaN,NaN,NaN,...,"Nevus, Atypical, Dysplastic, or Clark","Nevus, Dysplastic",NaN,NaN,dermoscopic,NaN,True,NaN,NaN,female
1,ISIC_0000001,Anonymous,CC-0,30.0,Trunk,Anterior trunk,Anterior abdomen,NaN,NaN,NaN,...,"Nevus, Atypical, Dysplastic, or Clark","Nevus, Dysplastic",NaN,NaN,dermoscopic,NaN,True,NaN,NaN,female
2,ISIC_0000002,Anonymous,CC-0,60.0,Upper extremity,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,histopathology,NaN,dermoscopic,NaN,True,NaN,NaN,female
3,ISIC_0000003,Anonymous,CC-0,30.0,Upper extremity,NaN,NaN,NaN,NaN,NaN,...,"Nevus, Atypical, Dysplastic, or Clark","Nevus, Dysplastic",NaN,NaN,dermoscopic,NaN,True,NaN,NaN,male
4,ISIC_0000004,Anonymous,CC-0,80.0,Trunk,Posterior trunk,NaN,NaN,NaN,NaN,...,NaN,NaN,histopathology,NaN,dermoscopic,NaN,True,NaN,NaN,male


In [88]:
isic_raw["attribution"].value_counts()

attribution
Hospital Clínic de Barcelona                                            12413
ViDIR Group, Department of Dermatology, Medical University of Vienna    10015
Anonymous                                                                2903
Name: count, dtype: int64

In [89]:
isic_raw["copyright_license"].value_counts()

copyright_license
CC-BY-NC    22428
CC-0         2903
Name: count, dtype: int64

In [90]:
isic_raw["diagnosis_1"].value_counts(dropna=False)

diagnosis_1
Benign           15990
Malignant         8473
Indeterminate      868
Name: count, dtype: int64

In [91]:
isic_raw["image_type"].value_counts()

image_type
dermoscopic    25331
Name: count, dtype: int64

In [92]:
isic_raw["lesion_id"].nunique()

12264

In [93]:
isic_raw["lesion_id"].isna().sum()

np.int64(1667)

In [94]:
on_disk = {p.name for p in (RAW / "isic").rglob("*.jpg")}
(isic_raw["isic_id"] + ".jpg").isin(on_disk).mean()   # fraction of register rows whose image exists

np.float64(1.0)

In [95]:
# Drop the indeterminate rows
isic = isic_raw[isic_raw["diagnosis_1"].isin(["Benign", "Malignant"])].copy()
print(len(isic))
isic["diagnosis_1"].value_counts()


24463


diagnosis_1
Benign       15990
Malignant     8473
Name: count, dtype: int64

In [96]:
isic[isic["lesion_id"].isna()]["attribution"].value_counts()

attribution
Anonymous    1666
Name: count, dtype: int64

In [97]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [98]:
import sys; sys.path.append("..")
from src.data.harmonise import prepare_fitzpatrick, prepare_ddi, prepare_isic

In [99]:
from pathlib import Path
print(Path("..").resolve())                  # which folder is ".." really?
print(sorted(p.name for p in Path("../src/data").iterdir()))   # what's actually in src/data?

/Users/alexandrekouakou/Downloads/Derma project/cap679-repo
['__init__.py', '__pycache__', 'dataset.py', 'harmonise.py']


In [100]:
from collections import Counter
Counter(str(p.parent) for p in (RAW / "fitzpatrick17k").rglob("*.jpg"))

Counter({'../data/raw/fitzpatrick17k': 16577})

In [101]:
fitz_clean = prepare_fitzpatrick(RAW / "fitzpatrick17k" / "fitzpatrick17ku.csv",
                                 RAW / "fitzpatrick17k")
fitz_clean.shape

raw rows:                                   16577
after dropping unknown skin:                16012
after keeping benign/malignant skin:        4320


(4320, 5)

In [102]:
print(fitz_clean["target"].mean())       # expect 0.5
pd.crosstab(fitz_clean["skin_group"], fitz_clean["target"]).reindex(["light", "medium", "dark"])

0.5


target,0,1
skin_group,,
light,1115,1195
medium,842,757
dark,203,208


In [103]:
Counter(str(p.parent) for p in (RAW / "ddi").rglob("*.png"))

Counter({'../data/raw/ddi': 656})

In [104]:
ddi_clean = prepare_ddi(RAW / "ddi" / "ddi_metadata.csv",
                                 RAW / "ddi")
ddi_clean.shape

ddi rows:                                   656


(656, 5)

In [105]:
print(ddi_clean["target"].mean())        # expect ≈ 0.26
pd.crosstab(ddi_clean["skin_group"], ddi_clean["target"]).reindex(["light", "medium", "dark"])

0.2606707317073171


target,0,1
skin_group,,
light,159,49
medium,167,74
dark,159,48


In [106]:
Counter(str(p.parent) for p in (RAW / "isic").rglob("*.jpg"))

Counter({'../data/raw/isic': 25331})

In [107]:
isic_clean = prepare_isic(RAW / "isic" / "metadata.csv",
                                 RAW / "isic")
isic_clean.shape

raw rows:                                   25331
after keeping benign/malignant skin:        24463


(24463, 5)

In [108]:
print(isic_clean["target"].mean())        # expect ≈ 0.346
isic_clean["group_id"].nunique()

0.34635980869067573


13606

In [109]:
all_data = pd.concat([fitz_clean, ddi_clean, isic_clean])
all_data.groupby("source").agg(images=("target", "size"),
                               malignant_rate=("target", "mean"))

,images,malignant_rate
source,,
ddi,656,0.260671
fitzpatrick17k,4320,0.500000
isic,24463,0.346360
